# Driver Drowsiness Detection - Model Comparison Experiment
## Objective
Compare **YOLO detectors** (v8, v9, v11, v12) and **CNN classifiers** to find the optimal combo.

- **YOLO**: Benchmarked on inference speed (no retraining needed)
- **CNN**: Loaded from trained weights in `models/weights/`

In [ ]:
import sys, time, json
from pathlib import Path
PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torchvision import datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## Part 1: Training Results Summary
Load results from `training_summary.json` generated by `train.py`.

In [ ]:
with open('models/results/training_summary.json') as f:
    training_summary = json.load(f)

rows = []
for name, data in training_summary.items():
    wt = Path(f'models/weights/{name}_best.pt')
    size_mb = round(wt.stat().st_size / 1e6, 1) if wt.exists() else 'N/A'
    rows.append({
        'Model': name,
        'Best Val Acc (%)': round(data['best_val_acc'] * 100, 2),
        'Best Epoch': data['best_epoch'],
        'Train Time (s)': round(data['training_time_s'], 1),
        'Weight Size (MB)': size_mb,
    })

df_summary = pd.DataFrame(rows).sort_values('Best Val Acc (%)', ascending=False)
print('=== CNN Training Results (from trained weights) ===')
print(df_summary.to_string(index=False))

In [ ]:
# Bar chart of validation accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#e63757','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD']
names = df_summary['Model'].tolist()
accs = df_summary['Best Val Acc (%)'].tolist()
sizes = [float(s) for s in df_summary['Weight Size (MB)'].tolist()]

ax1.barh(names, accs, color=colors[:len(names)])
ax1.set_xlabel('Validation Accuracy (%)')
ax1.set_title('CNN Accuracy Comparison (higher = better)')
ax1.set_xlim(40, 105)
for i, v in enumerate(accs): ax1.text(v+0.5, i, f'{v:.1f}%', va='center', fontweight='bold')

ax2.barh(names, sizes, color=colors[:len(names)])
ax2.set_xlabel('Model Size (MB)')
ax2.set_title('Weight File Size (lower = lighter)')
for i, v in enumerate(sizes): ax2.text(v+0.5, i, f'{v:.1f}MB', va='center')

plt.tight_layout()
plt.savefig('models/results/cnn_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2: CNN Test Set Evaluation
Load best weights and evaluate on the held-out test set.

In [ ]:
from src.classification.model_builder import SUPPORTED_MODELS, build_model
from src.utils.preprocessing import get_val_transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load test set
test_ds = datasets.ImageFolder('data/processed/test', transform=get_val_transforms(224))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
print(f'Test set: {len(test_ds)} images | Classes: {test_ds.classes}')

In [ ]:
test_results = []
for name in SUPPORTED_MODELS:
    wt = Path(f'models/weights/{name}_best.pt')
    if not wt.exists():
        print(f'{name}: weights not found, skipping'); continue
    model = build_model(name, num_classes=2, pretrained=False).to(device)
    model.load_state_dict(torch.load(str(wt), map_location=device, weights_only=True))
    model.eval()
    correct, total, all_preds, all_labels = 0, 0, [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            _, pred = out.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = correct / total * 100
    test_results.append({'Model': name, 'Test Acc (%)': round(acc, 2), 'preds': all_preds, 'labels': all_labels})
    print(f'{name:<18} | Test Accuracy: {acc:.2f}%')

print('\n=== Test Set Results ===')
print(pd.DataFrame([{k:v for k,v in r.items() if k != 'preds' and k != 'labels'} for r in test_results]).to_string(index=False))

In [ ]:
# Confusion matrices for top 3 models
top3 = sorted(test_results, key=lambda x: x['Test Acc (%)'], reverse=True)[:3]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, res in zip(axes, top3):
    cm = confusion_matrix(res['labels'], res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Alert','Drowsy'],
                yticklabels=['Alert','Drowsy'], ax=ax)
    ax.set_title(f"{res['Model']} ({res['Test Acc (%)']:.1f}%)")
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices - Top 3 Models', fontsize=14)
plt.tight_layout()
plt.savefig('models/results/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: CNN Inference Speed Benchmark

In [ ]:
N_ITERS = 50
dummy = torch.randn(1, 3, 224, 224, device=device)
cnn_speed = []
for name in SUPPORTED_MODELS:
    wt = Path(f'models/weights/{name}_best.pt')
    if not wt.exists(): continue
    model = build_model(name, num_classes=2, pretrained=False).to(device).eval()
    model.load_state_dict(torch.load(str(wt), map_location=device, weights_only=True))
    with torch.no_grad():
        for _ in range(5): model(dummy)  # warmup
    times = []
    with torch.no_grad():
        for _ in range(N_ITERS):
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    avg = np.mean(times)
    test_acc = next((r['Test Acc (%)'] for r in test_results if r['Model'] == name), 0)
    cnn_speed.append({'Model': name, 'Avg (ms)': round(avg, 2), 'FPS': round(1000/avg, 1),
                      'Size (MB)': round(wt.stat().st_size/1e6, 1), 'Test Acc (%)': test_acc})
    print(f'{name:<18} | {avg:.2f}ms | {1000/avg:.0f} FPS')

speed_df = pd.DataFrame(cnn_speed)
print('\n=== CNN Speed Comparison ===')
print(speed_df.to_string(index=False))

---
## Part 4: YOLO Face Detector Speed Benchmark
Benchmark pre-trained YOLO models on synthetic frames (no training needed).

In [ ]:
from ultralytics import YOLO

YOLO_MODELS = ['yolov8n.pt', 'yolov9c.pt', 'yolo11n.pt']
N_FRAMES, N_WARMUP = 20, 3
test_frames = [np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8) for _ in range(N_FRAMES)]

yolo_results = []
for model_name in YOLO_MODELS:
    print(f'Benchmarking {model_name}...')
    try:
        model = YOLO(model_name)
        for i in range(N_WARMUP): model.predict(test_frames[i], verbose=False)
        times = []
        for frame in test_frames:
            t0 = time.perf_counter()
            model.predict(frame, verbose=False, conf=0.4)
            times.append((time.perf_counter() - t0) * 1000)
        avg_ms = np.mean(times)
        yolo_results.append({'Model': model_name.replace('.pt',''), 'Avg (ms)': round(avg_ms, 2),
                             'FPS': round(1000/avg_ms, 1), 'Min (ms)': round(np.min(times), 2),
                             'Max (ms)': round(np.max(times), 2)})
        print(f'  {avg_ms:.1f}ms | {1000/avg_ms:.1f} FPS')
    except Exception as e:
        print(f'  FAILED: {e}')

yolo_df = pd.DataFrame(yolo_results)
print('\n=== YOLO Detector Comparison ===')
print(yolo_df.to_string(index=False))

In [ ]:
# YOLO speed chart
fig, ax = plt.subplots(figsize=(10, 4))
names = [r['Model'] for r in yolo_results]
fps = [r['FPS'] for r in yolo_results]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
ax.barh(names, fps, color=colors[:len(names)])
ax.set_xlabel('FPS'); ax.set_title('YOLO Detector Speed (higher = better)')
for i, v in enumerate(fps): ax.text(v+0.5, i, f'{v:.0f} FPS', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('models/results/yolo_speed.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Combined Pipeline Speed (YOLO + CNN)
Estimate total latency for every YOLO + CNN combo.

In [ ]:
combo = []
for yr in yolo_results:
    for cr in cnn_speed:
        total = yr['Avg (ms)'] + cr['Avg (ms)']
        combo.append({'YOLO': yr['Model'], 'CNN': cr['Model'], 'YOLO (ms)': yr['Avg (ms)'],
                      'CNN (ms)': cr['Avg (ms)'], 'Total (ms)': round(total, 2),
                      'Pipeline FPS': round(1000/total, 1), 'Test Acc (%)': cr['Test Acc (%)']})
combo_df = pd.DataFrame(combo).sort_values('Total (ms)')
print('=== Top 10 Fastest Combos ===')
print(combo_df.head(10).to_string(index=False))
print(f'\nFastest: {combo_df.iloc[0]["YOLO"]} + {combo_df.iloc[0]["CNN"]} @ {combo_df.iloc[0]["Pipeline FPS"]} FPS')

In [ ]:
# Pipeline heatmap
pivot = combo_df.pivot(index='CNN', columns='YOLO', values='Pipeline FPS')
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_title('Pipeline FPS: YOLO + CNN (higher = better)', fontsize=13)
ax.set_xlabel('YOLO Detector'); ax.set_ylabel('CNN Classifier')
plt.colorbar(im, label='FPS')
plt.tight_layout()
plt.savefig('models/results/pipeline_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 6: Accuracy vs Speed Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for r in cnn_speed:
    ax.scatter(r['Avg (ms)'], r['Test Acc (%)'], s=r['Size (MB)']*3, alpha=0.7, edgecolors='white', linewidth=2)
    ax.annotate(r['Model'], (r['Avg (ms)']+0.3, r['Test Acc (%)']+0.3), fontsize=10, fontweight='bold')
ax.set_xlabel('Inference Time (ms) - lower is better', fontsize=12)
ax.set_ylabel('Test Accuracy (%) - higher is better', fontsize=12)
ax.set_title('CNN Models: Accuracy vs Speed (bubble size = model weight)', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('models/results/accuracy_vs_speed.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 7: EAR/MAR Sensitivity Analysis

In [ ]:
from src.utils.drowsiness_utils import compute_drowsiness_score
ear_range = np.linspace(0.10, 0.40, 30)
cnn_range = np.linspace(0.0, 1.0, 30)
score_grid = np.zeros((len(cnn_range), len(ear_range)))
for i, cnn in enumerate(cnn_range):
    for j, ear in enumerate(ear_range):
        score_grid[i, j] = compute_drowsiness_score(ear=ear, mar=0.4, cnn_confidence=cnn)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.contourf(ear_range, cnn_range, score_grid, levels=20, cmap='RdYlGn_r')
plt.colorbar(im, label='Drowsiness Score')
ax.set_xlabel('EAR (Eye Aspect Ratio)', fontsize=12)
ax.set_ylabel('CNN Drowsy Confidence', fontsize=12)
ax.set_title('Drowsiness Score Heatmap', fontsize=14)
ax.axvline(0.25, color='red', linestyle='--', label='EAR Threshold')
ax.axhline(0.65, color='blue', linestyle='--', label='CNN Threshold')
ax.legend()
plt.tight_layout()
plt.savefig('models/results/drowsiness_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Conclusion & Recommendation

| Use Case | YOLO | CNN | Reason |
|----------|------|-----|--------|
| **Max Accuracy** | YOLO11n | ResNet50 | 100% val accuracy |
| **Best Balanced** | YOLO11n | MobileNetV2 | 97.2% acc, fast, 8.7MB |
| **Edge/Mobile** | YOLOv8n | MobileNetV2 | Smallest + fastest combo |